# T3-05 · Chọn silver pool và chuẩn bị bundles
Chạy trên Colab CPU. Notebook chọn ID, xác thực MS1, tải đúng 5 MS2_RESULTS.zip theo Drive file ID vào ổ tạm Colab và đọc ZIP trực tiếp, không giải nén. Chưa gán nhãn.


In [ ]:
# @title 1. Kết nối Drive và lấy code từ nhánh T3-05
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
from pathlib import Path
REPO = Path('/content/arxiv-data-science')
BRANCH = 'feature/m3-tv3-auto-labeling'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
import arxiv_pipeline
print('Code:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# @title 2. Chọn 2.000 bài và danh sách dự phòng
from pathlib import Path
import sys
REPO = Path('/content/arxiv-data-science')
if not (REPO/'src/arxiv_pipeline').is_dir():
    raise RuntimeError('Chưa có code T3-05 trong Colab. Hãy chạy cell 1 trước.')
sys.path.insert(0, str(REPO/'src'))
from arxiv_pipeline.matching.silver_pool import select_pool
POPULATION = Path('/content/drive/MyDrive/T3-05/population_manifest.json') # @param {type:'string'}
OUTPUT = Path('/content/drive/MyDrive/T3-05/output') # @param {type:'string'}
TARGET_COUNT = 2000 # @param {type:'integer'}
SEED = 42 # @param {type:'integer'}
selection = select_pool(Path(POPULATION), REPO/'configs/ms1_group6_ranges.json', Path(OUTPUT)/'selection.json', seed=SEED, target_count=TARGET_COUNT)
print('Đã chọn:', sum(map(len, selection['selected'].values())), '| quota:', selection['minimum_auto_count'])
print('Dự phòng:', sum(map(len, selection['reserves'].values())))


In [ ]:
# @title 3. Kiểm tra MS1, lấy đúng 5 ZIP MS2 và tạo bundles
import hashlib, json, shutil, zipfile
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from arxiv_pipeline.matching.silver_pool import prepare_pool
from arxiv_pipeline.matching.io import ZipSnapshot
MS1_PARENT = Path('/content/drive/MyDrive/arxiv_ms1_data') # @param {type:'string'}
INVENTORY = Path('/content/drive/MyDrive/T3-05/source_inventory.json')
CACHE = Path('/content/t3-05-ms2-zips')
ROOTS = Path(OUTPUT)/'roots.json'
assert INVENTORY.is_file(), f'Thiếu source_inventory.json: {INVENTORY}'
source = json.loads(INVENTORY.read_text(encoding='utf-8'))['members']
assert set(source) == {f'tv{i}' for i in range(1, 6)}
if not MS1_PARENT.is_dir():
    raise FileNotFoundError(f'Colab không thấy {MS1_PARENT}. Kiểm tra shortcut MyDrive của arxiv_ms1_data.')
ms1_roots = {}
for member in sorted(source):
    matches = [p for p in MS1_PARENT.iterdir() if p.is_dir() and p.name.casefold() == member]
    if len(matches) != 1:
        raise FileNotFoundError(f'Cần đúng một folder {member} trong {MS1_PARENT}; tìm thấy {len(matches)}')
    root = matches[0]/'data'
    if not (root/'state').is_dir() or not (root/'references').is_dir():
        raise FileNotFoundError(f'MS1 thiếu state/ hoặc references/: {root}')
    ms1_roots[member] = root
print('MS1: đã thấy đủ 5 folder data/')
auth.authenticate_user()
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
metadata = {}
for member, item in sorted(source.items()):
    file_id = item['ms2_results_zip_id']
    info = drive_api.files().get(fileId=file_id, fields='id,name,size,md5Checksum', supportsAllDrives=True).execute()
    if info.get('name') != 'MS2_RESULTS.zip' or int(info.get('size', 0)) <= 0:
        raise ValueError(f'{member}: ID {file_id} không trỏ tới MS2_RESULTS.zip hợp lệ: {info}')
    metadata[member] = info
    print(member, info['name'], f"{int(info['size'])/1e9:.2f} GB", file_id)
def md5_file(path):
    h = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8*1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
cache_ok = {}
for member, info in metadata.items():
    target = CACHE/member/'MS2_RESULTS.zip'
    expected_md5 = info.get('md5Checksum')
    cache_ok[member] = (target.is_file() and target.stat().st_size == int(info['size'])
                        and (not expected_md5 or md5_file(target) == expected_md5))
needed = sum(int(info['size']) for member, info in metadata.items() if not cache_ok[member])
if shutil.disk_usage('/content').free < needed + 2_000_000_000:
    raise OSError(f'Ổ tạm Colab thiếu chỗ: cần khoảng {needed/1e9+2:.1f} GB')
roots = {}
for member, info in sorted(metadata.items()):
    target = CACHE/member/'MS2_RESULTS.zip'
    target.parent.mkdir(parents=True, exist_ok=True)
    expected_md5 = info.get('md5Checksum')
    if not cache_ok[member]:
        partial = target.with_suffix('.partial')
        if partial.exists(): partial.unlink()
        request = drive_api.files().get_media(fileId=info['id'], supportsAllDrives=True)
        with partial.open('wb') as handle:
            downloader = MediaIoBaseDownload(handle, request, chunksize=32*1024*1024)
            done = False
            while not done:
                progress, done = downloader.next_chunk(num_retries=3)
                print(f'{member}: {progress.progress():.0%}', end='\r')
        if partial.stat().st_size != int(info['size']) or (expected_md5 and md5_file(partial) != expected_md5):
            raise ValueError(f'{member}: ZIP tải về không khớp size/MD5 Drive')
        partial.replace(target)
    with ZipSnapshot(target) as check:
        print(member, 'ZIP OK; root:', check.prefix or '<archive root>')
    roots[member] = {'ms1_root': str(ms1_roots[member]), 'ms2_root': str(target)}
ROOTS.parent.mkdir(parents=True, exist_ok=True)
ROOTS.write_text(json.dumps(roots, indent=2), encoding='utf-8')
result = prepare_pool(Path(OUTPUT)/'selection.json', ROOTS, Path(OUTPUT)/'prepared')
print('Ready:', result['ready'], '| Chưa ready:', result['not_ready'])
print('Manifest:', Path(OUTPUT)/'prepared/prepared_manifest.json')
